### Import Libraries

In [1]:
# Import required libraries for KNN classifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Libraries imported successfully!


### Load Dataset

In [2]:
# Load house prediction dataset (same as Task 2)
df = pd.read_csv('4) house Prediction Data Set.csv', sep='\s+')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Dataset shape: (505, 14)

First 5 rows:


,0.00632,18.00,2.310,0,0.5380,6.5750,65.20,4.0900,1,296.0,15.30,396.90,4.98,24.00
0,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
1,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
2,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
3,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
4,0.02985,0.0,2.18,0,0.458,6.430,58.7,6.0622,3,222.0,18.7,394.12,5.21,28.7


### Create Classification Target (Bin Continuous Prices)

In [3]:
# Convert continuous house price target to 3 classification categories (low/medium/high)
target_col = df.columns[-1]  # Last column is the price target
X = df.drop(target_col, axis=1)
y = pd.qcut(df[target_col], q=3, labels=[0, 1, 2])  # 0=Low, 1=Medium, 2=High

print("Target distribution (classification bins):")
print(y.value_counts())
print(f"\nFeatures shape: {X.shape}, Target shape: {y.shape}")

Target distribution (classification bins):
24.00
1    171
0    169
2    165
Name: count, dtype: int64

Features shape: (505, 13), Target shape: (505,)


### Preprocess Data

In [4]:
# Handle missing values for numerical features
X = X.fillna(X.mean())

# Split into train/test (80-20 split, stratify to preserve class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (KNN is distance-based, scaling is critical)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=== Preprocessing Complete ===")
print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Testing set: {X_test_scaled.shape[0]} samples")

=== Preprocessing Complete ===
Training set: 404 samples
Testing set: 101 samples


### Train KNN with Different K Values

In [5]:
# Test different K values and compare performance
k_values = [3, 5, 7, 9, 11]
results = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    
    results.append({
        'K': k,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4)
    })

print("=== K Value Comparison ===")
print(pd.DataFrame(results))

=== K Value Comparison ===
    K  Accuracy  Precision  Recall
0   3    0.7624     0.7771  0.7624
1   5    0.7822     0.8010  0.7822
2   7    0.7822     0.7998  0.7822
3   9    0.8020     0.8197  0.8020
4  11    0.7624     0.7732  0.7624


### Evaluate Best K Model

In [6]:
# Select best K (highest accuracy)
results_df = pd.DataFrame(results)
best_k = results_df.loc[results_df['Accuracy'].idxmax(), 'K']
print(f"Best K value: {best_k}")

# Train final model with best K
final_model = KNeighborsClassifier(n_neighbors=int(best_k))
final_model.fit(X_train_scaled, y_train)
y_pred_final = final_model.predict(X_test_scaled)

print("\n=== Final Model Evaluation ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Weighted Precision: {precision_score(y_test, y_pred_final, average='weighted'):.4f}")
print(f"Weighted Recall: {recall_score(y_test, y_pred_final, average='weighted'):.4f}")

Best K value: 9

=== Final Model Evaluation ===
Accuracy: 0.8020
Weighted Precision: 0.8197
Weighted Recall: 0.8020


### Confusion Matrix & Classification Report

In [7]:
print("Confusion Matrix (Rows=Actual, Columns=Predicted):")
print(confusion_matrix(y_test, y_pred_final))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=['Low', 'Medium', 'High']))

Confusion Matrix (Rows=Actual, Columns=Predicted):
[[28  6  0]
 [ 3 29  2]
 [ 2  7 24]]

Classification Report:
              precision    recall  f1-score   support

         Low       0.85      0.82      0.84        34
      Medium       0.69      0.85      0.76        34
        High       0.92      0.73      0.81        33

    accuracy                           0.80       101
   macro avg       0.82      0.80      0.80       101
weighted avg       0.82      0.80      0.80       101

